<a href="https://colab.research.google.com/github/Chathushi2004/Statical-Learning-e23309/blob/main/Copy_of_data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
import io
from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder

In [ ]:
class PlottingMethods:
    """Handles granular chart generation returning Plotly figures."""

    @staticmethod
    def create_bar(df, column, title):
        """Generates a bar chart with raw counts and percentages."""
        counts = df[column].value_counts().reset_index()
        counts.columns = [column, 'count']
        counts['percentage'] = (counts['count'] / counts['count'].sum() * 100).round(2)

        fig = px.bar(counts, x=column, y='count', text=counts['percentage'].apply(lambda x: f'{x}%'),
                     title=title, labels={'count': 'Frequency'})
        return fig

    @staticmethod
    def create_histogram(df, column):
        return px.histogram(df, x=column, title=f"Distribution of {column}")

    @staticmethod
    def create_box(df, x_col, y_col):
        return px.box(df, x=x_col, y=y_col, points="all", title=f"{y_col} by {x_col}")

In [ ]:
class DataInspector:
    def __init__(self):
        self.df = None
        self.plotter = PlottingMethods()

    # --- 1. Data Ingestion & Sanitization ---
    def upload_data(self):
        """Handles local file uploads and sanitizes garbage strings."""
        uploaded = files.upload()
        for name, content in uploaded.items():
            # Convert garbage strings to NaN during load
            self.df = pd.read_csv(io.BytesIO(content),
                                  na_values=['?', 'n/a', 'NULL', ' ', 'nan'])

        # Auto-Type Correction
        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if not converted.isna().all():
                self.df[col] = converted
        print("Data loaded and sanitized successfully.")

    # --- 2. Structural Analysis & Cleaning ---
    def data_summary(self):
        if self.df is None: return
        print(f"Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}")
        print("\nColumn Breakdown:")
        print(self.df.dtypes.value_counts())
        return self.df.head(20)

    def handle_missing_values(self, strategy='mean', columns=None, fill_value=None):
        cols = columns if columns else self.df.columns
        for col in cols:
            if strategy == 'mean':
                self.df[col] = self.df[col].fillna(self.df[col].mean())
            elif strategy == 'median':
                self.df[col] = self.df[col].fillna(self.df[col].median())
            elif strategy == 'mode':
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            elif strategy == 'constant':
                self.df[col] = self.df[col].fillna(fill_value)

    def handle_outliers(self, column, action='flag'):
        """IQR-based outlier detection."""
        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        if action == 'delete':
            self.df = self.df[(self.df[column] >= lower) & (self.df[column] <= upper)]
        else:
            self.df[f'{column}_outlier'] = ~self.df[column].between(lower, upper)

    # --- 3. Feature Engineering (Normalization) ---
    def extract_normalized_numeric_data(self, method='standard'):
        nums = self.df.select_dtypes(include=[np.number])
        scaler = {'minmax': MinMaxScaler(), 'standard': StandardScaler(), 'robust': RobustScaler()}[method]
        return pd.DataFrame(scaler.fit_transform(nums), columns=nums.columns)

    # --- 4. Advanced Visualization ---
    def univariate_subplots(self, column):
        """3-panel subplot: Violin/Box, Scatter, Histogram."""
        fig = make_subplots(rows=1, cols=3, subplot_titles=("Box/Violin", "Index vs Value", "Histogram"))

        fig.add_trace(go.Violin(y=self.df[column], box_visible=True, points='all'), row=1, col=1)
        fig.add_trace(go.Scatter(y=self.df[column], mode='markers'), row=1, col=2)
        fig.add_trace(go.Histogram(x=self.df[column]), row=1, col=3)
        fig.show()

    def plot_relationship(self, col1, col2):
        """Smart relationship detector."""
        t1, t2 = self.df[col1].dtype, self.df[col2].dtype

        # Num-Num
        if np.issubdtype(t1, np.number) and np.issubdtype(t2, np.number):
            return px.scatter(self.df, x=col1, y=col2, trendline="ols")
        # Cat-Num
        elif np.issubdtype(t1, np.object_) != np.issubdtype(t2, np.number):
            return px.box(self.df, x=col1, y=col2, points="all")
        # Cat-Cat
        else:
            return px.histogram(self.df, x=col1, color=col2, barmode='group')

    # --- 5. Deep Statistical Insights ---
    def plot_all_associations_heatmap(self):
        """Cramer's V and Pearson Heatmap."""
        cols = self.df.columns
        matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

        for c1 in cols:
            for c2 in cols:
                if c1 == c2:
                    matrix.loc[c1, c2] = 1.0
                else:
                    # Simplified logic: use Pearson for numeric-numeric
                    if np.issubdtype(self.df[c1].dtype, np.number) and np.issubdtype(self.df[c2].dtype, np.number):
                        matrix.loc[c1, c2] = self.df[c1].corr(self.df[c2])
                    else:
                        # Placeholder for Cramer's V / Mixed (Set to 0 for brevity in example)
                        matrix.loc[c1, c2] = 0

        fig = px.imshow(matrix, text_auto=True, title="Unified Association Heatmap")
        fig.show()

In [21]:
# 1. Initialize
inspector = DataInspector()

# 2. Upload (Select titanic.csv from your computer)
inspector.upload_data()

# 3. Summary
inspector.data_summary()

# 4. Cleaning
inspector.handle_missing_values(strategy='median', columns=['Age'])
inspector.handle_outliers('Fare', action='delete')

# 5. Visualize
# Example: Smart Relationship (Numeric vs Categorical)
fig = inspector.plot_relationship('Survived', 'Age')
fig.show()

# Example: Univariate Subplot
inspector.univariate_subplots('Age')

# 6. Heatmap
inspector.plot_all_associations_heatmap()

Saving titanic.csv to titanic.csv
Data loaded and sanitized successfully.
Rows: 891 | Columns: 12

Column Breakdown:
int64      5
object     4
float64    3
Name: count, dtype: int64
